# Pre-label bằng SAM3 (text-prompt) trên Kaggle — xuất COCO JSON để import vào CVAT

**Mục đích:** thay vì vẽ box tay từ đầu cho toàn bộ ảnh, dùng SAM3 (model segmentation mạnh nhất hiện có, hỗ trợ prompt bằng text như "helmet", "license plate"...) để tự động khoanh vùng object trước — bạn chỉ cần **sửa lại** trong CVAT (xoá box sai, sửa box lệch, gán đúng label) thay vì vẽ từ đầu.

**Giới hạn cần biết trước:**
- SAM3 khoanh được vùng "chân người" nhưng KHÔNG tự phân biệt được đang lái (`Leg Riding`) hay đang dắt bộ (`Leg Standing`) — notebook này gán mặc định tất cả là `Leg Riding`, bạn cần tự đổi label những case dắt bộ trong CVAT sau khi import.
- `No Plate` (xe không gắn biển) KHÔNG auto-label được vì đây là sự vắng mặt của object — cần tự đánh dấu tay hoàn toàn trong CVAT.

**Trước khi chạy (bắt buộc):**
1. **Settings > Accelerator > GPU T4 x2**.
2. **Settings > Internet > On**.
3. Nén 3 thư mục `datasets/cvat_review/helmet_review`, `plate_review`, `new_classes_pool` thành 3 file zip riêng, **+ Add Input > Upload** từng file, đặt tên dataset đúng như biến `*_DATASET_NAME` ở ô code bên dưới (hoặc sửa biến cho khớp tên bạn đặt) — không cần quan tâm path chính xác Kaggle mount vào đâu, notebook tự dò.
4. Điền `HF_TOKEN` (xem hướng dẫn lấy token ở dưới) — `SAM3_MODEL_ID` đã điền sẵn `facebook/sam3`.
5. **Chạy cell "fix môi trường" (cell ngay sau ô config) → RESTART SESSION khi cell đó bảo → rồi mới chạy tiếp các cell còn lại (không phải 1 lần "Run All" duy nhất — pip cài numpy/scipy mới bắt buộc phải restart mới áp dụng được, đây là quy trình bình thường, không phải lỗi).**

**Kết quả:** 3 file COCO JSON trong `/kaggle/working/` — tải qua tab **Output**, import từng file vào đúng CVAT task tương ứng (Task → Actions → Upload annotations → format **COCO 1.0**).

**Lấy HF Token:** vào [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **Create new token** → loại **Read** → copy dán vào `HF_TOKEN`.

**Nếu lỗi:** báo lại traceback đầy đủ (không chỉ dòng cuối) để chẩn đoán đúng chỗ.

In [ ]:
HELMET_DATASET_NAME = "helmet-review"       # tên dataset bạn đặt lúc Add Input, chứa ảnh datasets/cvat_review/helmet_review
PLATE_DATASET_NAME = "plate-review"         # tên dataset bạn đặt lúc Add Input, chứa ảnh datasets/cvat_review/plate_review
NEWCLASS_DATASET_NAME = "new-classes-pool"  # tên dataset bạn đặt lúc Add Input, chứa ảnh datasets/cvat_review/new_classes_pool

# Ngưỡng confidence tối thiểu để giữ lại 1 detection — pre-label chỉ cần "gợi ý",
# để thấp một chút (0.25) rồi bạn lọc/xoá box sai trong CVAT còn nhanh hơn là bỏ sót.
CONF_THRESHOLD = 0.25

# --- HuggingFace SAM3 (đã được accept quyền truy cập model gated) ---
# Dán token HF của bạn (huggingface.co/settings/tokens, tạo loại "Read") — dùng Kaggle
# Secrets (Add-ons > Secrets) thay vì dán thẳng vào đây nếu không muốn lộ token trong notebook.
HF_TOKEN = ""  # BẠN tự điền — vd: "hf_xxxxxxxxxxxxxxxxxxxx"

# facebook/sam3 — đã tra chính thức trên huggingface.co/docs/transformers/model_doc/sam3,
# đây là model id thật, không cần đổi trừ khi bạn được accept 1 checkpoint khác (vd: sam3.1).
SAM3_MODEL_ID = "facebook/sam3"

In [ ]:
# Dùng HuggingFace transformers thay vì repo git của Meta — quản lý dependency tốt hơn
# nhiều (không tự ý ép numpy<2 làm vỡ torch/torchvision/opencv có sẵn như bản git đã dính
# lỗi "numpy.dtype size changed" trước đó).
#
# LỖI THẬT ĐÃ GẶP TRÊN COLAB (môi trường Kaggle base image cũng cài numba sẵn tương tự nên
# fix chung luôn ở đây): "NumPy / Numba binary conflict" — máy ảo cài sẵn numba biên dịch
# khớp với numpy gốc; khi ta nâng numpy lên bản mới hơn cho transformers/SAM3, numba cũ
# không còn đọc được ABI numpy mới → lỗi dù notebook này không hề import numba trực tiếp
# (gói khác âm thầm import nó). Fix: GỠ HẲN numba (không dùng tới) thay vì cố ép version
# khớp nhau, rồi cài numpy+scipy CÙNG LÚC trong 1 lệnh để pip tự chọn 1 cặp ABI khớp nhau
# (cài numpy rồi cài scipy riêng — 2 lệnh tách nhau — không đảm bảo pip khớp ABI, đã dính
# lỗi kiểu "dtype size changed"/"cannot import _slice"/"ufunc no __module__" vì lý do này).
#
# CHỈ 1 CELL DUY NHẤT LÀM VIỆC NÀY — không thêm cell cài lại numpy/scipy ở chỗ khác.
!pip uninstall -y numba sam3 2>/dev/null || true
!pip install -q -U --force-reinstall numpy scipy
!pip install -q -U transformers accelerate safetensors huggingface_hub

print("Đã cài xong.")
print("BẮT BUỘC: Restart Session NGAY BÂY GIỜ (Kaggle: nút Restart trên toolbar) — pip cài")
print("xong không tự áp dụng cho module đã import sẵn trong kernel đang chạy. Sau khi")
print("restart, chạy tiếp TỪ CELL PHÍA DƯỚI cell này trở đi — không chạy lại cell này.")

In [ ]:
# --- Load SAM3 qua HuggingFace transformers ---
# API đã verify thật từ huggingface.co/docs/transformers/model_doc/sam3 (không còn đoán):
# Sam3Model + Sam3Processor, gọi processor(images=.., text=..) rồi post-process bằng
# processor.post_process_instance_segmentation(...) — trả về dict có "boxes" (xyxy, pixel
# tuyệt đối) + "scores". Model dùng ảnh input 1008x1008 mặc định, không cần chỉnh gì thêm.
import torch
from transformers import Sam3Model, Sam3Processor

assert HF_TOKEN, "Điền HF_TOKEN ở ô config phía trên trước (hoặc dùng Kaggle Secrets)."

model = Sam3Model.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN, device_map="auto")
processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN)
model.eval()
print("device:", model.device)

def predict_boxes(image_path: str, text_prompt: str, conf_threshold: float = CONF_THRESHOLD):
    """Trả về list (x1, y1, x2, y2, score) cho 1 text prompt trên 1 ảnh."""
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=conf_threshold,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]
    boxes = []
    for box, score in zip(results["boxes"], results["scores"]):
        x1, y1, x2, y2 = [float(v) for v in box]
        boxes.append((x1, y1, x2, y2, float(score)))
    return boxes

In [ ]:
import os, json, glob
from PIL import Image  # KHÔNG dùng cv2 — opencv-python cài sẵn trong máy build cho numpy>=2,
                        # còn sam3 (bản git) ép numpy<2, hai bên đọc struct dtype khác size ->
                        # ValueError "numpy.dtype size changed". PIL không đụng numpy nên tránh
                        # được lỗi này (đã đổi sang HuggingFace transformers nên hết cần lo vụ
                        # này nữa, nhưng giữ PIL vì đơn giản, không có lý do đổi lại cv2).

def find_dataset_dir(dataset_name: str) -> str:
    """
    Tự dò đường dẫn thật của dataset đã mount trong /kaggle/input/ — không đoán cứng 1 kiểu
    path, vì Kaggle có thể mount theo "/kaggle/input/<name>/" HOẶC
    "/kaggle/input/datasets/<username>/<name>/" tuỳ cách bạn add input. Tìm theo TÊN THƯ MỤC
    khớp dataset_name ở bất kỳ độ sâu nào dưới /kaggle/input/, tránh lặp lại lỗi path sai
    từng gặp trước đó.
    """
    candidates = [d for d in glob.glob(f"/kaggle/input/**/{dataset_name}", recursive=True)
                  if os.path.isdir(d)]
    direct = f"/kaggle/input/{dataset_name}"
    if os.path.isdir(direct) and direct not in candidates:
        candidates.insert(0, direct)
    if not candidates:
        existing = glob.glob("/kaggle/input/*")
        raise FileNotFoundError(
            f"Không tìm thấy dataset '{dataset_name}' trong /kaggle/input/. "
            f"Kiểm tra lại tên bạn đặt lúc Add Input có đúng '{dataset_name}' không. "
            f"Hiện /kaggle/input/ có: {existing}"
        )
    return candidates[0]

def build_coco(dataset_name: str, prompt_to_label: dict, out_path: str):
    """
    Chạy SAM3 với từng (text_prompt -> label) trên toàn bộ ảnh trong dataset đã upload
    (tự dò path qua find_dataset_dir), ghi kết quả ra 1 file COCO JSON (categories = các
    label trong prompt_to_label).
    """
    image_dir = find_dataset_dir(dataset_name)
    print(f"Dùng thư mục: {image_dir}")

    categories = [{"id": i + 1, "name": label, "supercategory": "none"}
                  for i, label in enumerate(sorted(set(prompt_to_label.values())))]
    label_to_cat_id = {c["name"]: c["id"] for c in categories}

    images, annotations = [], []
    ann_id = 1
    files = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    print(f"{image_dir}: {len(files)} ảnh")
    if not files:
        raise FileNotFoundError(
            f"Thư mục '{image_dir}' tồn tại nhưng không có file .jpg nào bên trong. "
            f"Kiểm tra lại file zip đã upload có đúng ảnh không (glob debug): "
            f"{glob.glob(os.path.join(image_dir, '*'))[:10]}"
        )

    for img_id, fpath in enumerate(files, start=1):
        with Image.open(fpath) as im:
            w, h = im.size
        images.append({
            "id": img_id,
            "file_name": os.path.basename(fpath),
            "width": w,
            "height": h,
        })
        for prompt, label in prompt_to_label.items():
            for x1, y1, x2, y2, score in predict_boxes(fpath, prompt):
                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": label_to_cat_id[label],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],  # COCO: [x, y, width, height]
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": 0,
                    "score": score,
                })
                ann_id += 1
        if img_id % 20 == 0:
            print(f"  {img_id}/{len(files)}")

    coco = {"images": images, "annotations": annotations, "categories": categories}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f"Xong: {out_path} — {len(images)} ảnh, {len(annotations)} box")

## 1. helmet_review — With Helmet / Without Helmet

In [ ]:
build_coco(
    HELMET_DATASET_NAME,
    prompt_to_label={
        # Prompt phải nhắm thẳng vào VẬT THỂ (cái mũ), không mô tả qua người —
        # "person wearing a helmet" khiến SAM3 khoanh CẢ NGƯỜI thay vì chỉ cái mũ
        # (đã dính lỗi này thật, box ra to trùm hết cả thân người).
        "safety helmet": "With Helmet",
        "bare human head, no helmet": "Without Helmet",
    },
    out_path="/kaggle/working/helmet_review_sam3_coco.json",
)

## 2. plate_review — plate (No Plate không auto-label được, để trống tự đánh tay)

In [ ]:
build_coco(
    PLATE_DATASET_NAME,
    prompt_to_label={
        "vehicle license plate": "plate",
    },
    out_path="/kaggle/working/plate_review_sam3_coco.json",
)

## 3. new_classes_pool — Mirror

(Đã bỏ `Leg Riding`/`Leg Standing` khỏi notebook này — dự án dùng thẳng pose model có sẵn `yolov8n-pose` + tính góc hông-gối-mắt cá trong `app/cv/pose.py` để phân biệt riding/standing, đã test đúng trên video thật, không cần train thêm detector riêng cho phần này. Box tĩnh không thể hiện được góc khớp như skeleton, nên bỏ hướng này.)

In [ ]:
build_coco(
    NEWCLASS_DATASET_NAME,
    prompt_to_label={
        "motorcycle side mirror": "Mirror",
    },
    out_path="/kaggle/working/new_classes_pool_sam3_coco.json",
)

## 4. Tải kết quả về và import vào CVAT

Vào tab **Output** bên phải sau khi chạy xong, tải 3 file:
- `helmet_review_sam3_coco.json` → import vào task `helmet_review_v1`
- `plate_review_sam3_coco.json` → import vào task `plate_review_v1`
- `new_classes_pool_sam3_coco.json` → import vào task `new_classes_v1`

Trong CVAT, mở task tương ứng → menu 3 chấm → **Upload annotations** → format **COCO 1.0** → chọn đúng file JSON. Ảnh trong task PHẢI trùng tên với `file_name` trong JSON (đã khớp sẵn vì cùng lấy từ `datasets/cvat_review/`).

Sau khi import, CVAT sẽ hiện sẵn các box SAM3 đã đoán — bạn chỉ cần xoá box sai, sửa box lệch, và **đổi `Leg Riding` → `Leg Standing`** cho những trường hợp thực tế là dắt bộ.